In [4]:
%run nb_dq_utils

StatementMeta(, 952def97-bb73-43a8-bc26-ccb95e51c380, 20, Finished, Available, Finished, True)

In [5]:
from pyspark.sql import functions as F
 
logger = get_logger("silver.territory.dq")

# ── Read Bronze ───────────────────────────────────────────────────────
df_territory = spark.read.table(
    "lh_adventureworks_bronze.dbo.Sales_SalesTerritory"
)
logger.info(f"Territory rows: {df_territory.count()}")

expected_rows = df_territory.count()
# ── Select and standardize ────────────────────────────────────────────
df_silver_territory = df_territory.select(
    F.col("TerritoryID"),
    F.col("Name").alias("TerritoryName"),
    F.col("CountryRegionCode"),
    F.col("Group").alias("TerritoryGroup"),
    F.col("SalesYTD").cast("decimal(19,4)"),
    F.col("SalesLastYear").cast("decimal(19,4)"),
    F.col("CostYTD").cast("decimal(19,4)"),
    F.col("CostLastYear").cast("decimal(19,4)"),
    F.col("rowguid"),
    F.col("ModifiedDate")
)
df_silver_territory.cache()

StatementMeta(, 952def97-bb73-43a8-bc26-ccb95e51c380, 21, Finished, Available, Finished, False)

INFO:silver.territory.dq:Territory rows: 10


DataFrame[TerritoryID: int, TerritoryName: string, CountryRegionCode: string, TerritoryGroup: string, SalesYTD: decimal(19,4), SalesLastYear: decimal(19,4), CostYTD: decimal(19,4), CostLastYear: decimal(19,4), rowguid: string, ModifiedDate: timestamp]

In [6]:

actual_rows = df_silver_territory.count()
 
checks = [
    check_row_count(df_silver_territory, expected_rows),   # no manual dict needed —
                                                             # nothing later reuses actual_rows
    check_null_pk(df_silver_territory, "TerritoryID"),
    check_duplicate_pk(df_silver_territory, "TerritoryID"),
    # Reusing check_null_pk for a non-PK column — functionally identical to the
    # original custom check (same filter().isNull().count() logic), the only
    # difference is the resulting check name reads "Null PK — TerritoryName"
    # rather than "Null TerritoryName". Cosmetic trade-off, kept for consistency
    # with the rest of this notebook's checks.
    check_null_pk(df_silver_territory, "TerritoryName"),
]
 
run_dq_checks(checks, logger, "silver.territory")
 

StatementMeta(, 952def97-bb73-43a8-bc26-ccb95e51c380, 22, Finished, Available, Finished, False)

INFO:silver.territory.dq:[DQ] [PASS] Row count — Expected 10, got 10
INFO:silver.territory.dq:[DQ] [PASS] Null PK — TerritoryID — 0 nulls
INFO:silver.territory.dq:[DQ] [PASS] Duplicate PK — TerritoryID — 0 duplicates
INFO:silver.territory.dq:[DQ] [PASS] Null PK — TerritoryName — 0 nulls
INFO:silver.territory.dq:[DQ] All checks passed for silver.territory.


In [7]:
# ── Create schema and write ───────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.sales")
 
df_silver_territory.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_silver.sales.territory")

StatementMeta(, 952def97-bb73-43a8-bc26-ccb95e51c380, 23, Finished, Available, Finished, False)

In [8]:
# ── Verify ────────────────────────────────────────────────────────────
df_verify = spark.read.table("lh_adventureworks_silver.sales.territory")
logger.info(f"silver.sales.territory written — {df_verify.count():,} rows verified.")
 
df_silver_territory.unpersist()


StatementMeta(, 952def97-bb73-43a8-bc26-ccb95e51c380, 24, Finished, Available, Finished, False)

INFO:silver.territory.dq:silver.sales.territory written — 10 rows verified.


DataFrame[TerritoryID: int, TerritoryName: string, CountryRegionCode: string, TerritoryGroup: string, SalesYTD: decimal(19,4), SalesLastYear: decimal(19,4), CostYTD: decimal(19,4), CostLastYear: decimal(19,4), rowguid: string, ModifiedDate: timestamp]